# Does a Country's Electricity Mix Predict Its Air Pollution?

**The question:** Countries make electricity in different ways. Some burn coal, gas and oil (which release greenhouse gases and pollute the air). Others use solar, wind, hydro or nuclear power (which release far less). Can we predict how polluted a country's air is just by looking at how it makes its electricity?

**What this notebook does, in plain English:**

1. **Get the data** - download a dataset about how countries generate electricity, and combine it with a dataset on air pollution.
2. **Clean it up** - throw away incomplete or unreliable records and join the two datasets together.
3. **Teach a computer to make predictions** - we train a *machine learning model* on part of the data.
4. **Check its homework** - we test the model on data it has never seen and measure how well it did.

You can run the notebook from top to bottom by choosing **Run All** from the menu, or press **Shift + Enter** to run one cell at a time.

### Key terms used below
| Term | Meaning |
|---|---|
| **PM2.5** | Tiny airborne particles (smaller than 2.5 micrometres, much thinner than a hair) that are harmful to breathe. Measured in micrograms per cubic metre (µg/m³). Lower is better. |
| **GHG** | Greenhouse gas, such as CO₂, which traps heat in the atmosphere. |
| **DataFrame** | A table of data (like an Excel sheet) held in the computer's memory. |
| **Feature** | An input the model uses to make a prediction (here: the electricity mix). |
| **Target** | The thing we want to predict (here: PM2.5 pollution). |

## Step 0 - Load the tools we need

Programmers rarely write everything from scratch. They import **libraries**, which are ready-made toolkits. Here is what each one is for:

- **pandas** - works with tables of data.
- **numpy** - does maths.
- **matplotlib** - draws charts.
- **scikit-learn** (`sklearn`) - provides machine-learning models and ways to measure them.
- **kagglehub** - downloads datasets from Kaggle, a website that hosts public datasets.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Part 1 - Getting and cleaning the data

## Step 1 - Download the electricity data

We use the public **World Energy Consumption** dataset from Kaggle. It records, for every country and year, how much electricity came from each source (coal, solar, wind and so on). The code below downloads it automatically, so you do not need to fetch anything by hand.

The second dataset, **PM2.5 air pollution exposure**, is a file that already sits in the project folder (`average-exposure-pm25-pollution.csv`).

In [ ]:
# Download the energy dataset (this may take a moment the first time)
path = kagglehub.dataset_download("pralabhpoudel/world-energy-consumption")
energy_file = os.path.join(path, "World Energy Consumption.csv")

# The pollution file lives in the same folder as this notebook
pm25_file = "average-exposure-pm25-pollution.csv"

print("Energy data saved at:", energy_file)

## Step 2 - Define the cleaning recipe

Raw data is messy: some values are missing, and the two datasets name things differently. We write a **function**, which is a reusable recipe, that does the following:

1. **Read both files** into tables.
2. **Group the energy sources into two buckets:**
   - *Emitting* (coal, gas, oil): burning these releases greenhouse gases and pollutants.
   - *Non-emitting* (solar, wind, hydro, nuclear, other renewables): these release little or none.

   For each country and year we add up the percentages in each bucket. For example, a country might get 60% of its electricity from emitting sources and 40% from non-emitting sources.
3. **Keep only the columns we need:** country, year and those two totals.
4. **Rename columns** in the pollution table so both tables call the country and year columns by the same names. This lets us line them up.
5. **Merge** the tables, matching rows with the same country *and* year. Rows that exist in only one table are dropped (this is called an *inner* merge).
6. **Drop rows with missing values**, because a model cannot learn from blanks.
7. **Drop countries with too little history.** If a country has fewer than 15 usable years of data (`min_valid_years=15`), its numbers are too sparse to trust, so we remove it.
8. **Save the result** as `cleaned_climate_merged.csv` so we can reuse it later.

In [ ]:
def clean_and_merge_climate_data(energy_path, pm25_path, min_valid_years=15):
    energy_df = pd.read_csv(energy_path)
    pm25_df = pd.read_csv(pm25_path)

    # Which columns count as "emitting" and which as "non-emitting"?
    # (We only use columns that actually exist in the downloaded file.)
    emitting_cols = [c for c in ['coal_share_elec', 'gas_share_elec', 'oil_share_elec']
                     if c in energy_df.columns]
    clean_cols = [c for c in ['solar_share_elec', 'wind_share_elec', 'hydro_share_elec',
                              'nuclear_share_elec', 'other_renewables_share_elec']
                  if c in energy_df.columns]

    # Make a fresh copy of the table (avoids a harmless performance warning)
    energy_df = energy_df.copy()

    # Add up the percentages in each bucket, for every country and year
    energy_df['% GHG emitting source'] = energy_df[emitting_cols].sum(axis=1, min_count=1)
    energy_df['% GHG non emitting source'] = energy_df[clean_cols].sum(axis=1, min_count=1)

    # Keep only the columns we need
    energy_subset = energy_df[['country', 'year',
                               '% GHG emitting source', '% GHG non emitting source']]

    # Give the pollution table the same column names so the tables can be matched up
    pm25_df = pm25_df.rename(columns={
        'Entity': 'country',
        'Year': 'year',
        'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)': 'PM2.5 Exposure'
    })

    # Join the two tables on country + year, then drop rows with any blank value
    merged_df = pd.merge(energy_subset, pm25_df[['country', 'year', 'PM2.5 Exposure']],
                         on=['country', 'year'], how='inner')
    clean_merged = merged_df.dropna(subset=['% GHG emitting source',
                                            '% GHG non emitting source',
                                            'PM2.5 Exposure'])

    # Remove countries that have fewer than `min_valid_years` years of data
    valid_years_per_country = clean_merged.groupby('country').size()
    countries_to_keep = valid_years_per_country[valid_years_per_country >= min_valid_years].index

    final_df = clean_merged[clean_merged['country'].isin(countries_to_keep)]
    final_df.to_csv('cleaned_climate_merged.csv', index=False)

    print(f"Data cleaned successfully! Retained {len(countries_to_keep)} countries/regions.")
    return final_df

## Step 3 - Run the cleaning recipe

Defining a function only writes the recipe down; this cell actually **cooks** it. It also prints how many countries and regions survived the cleaning.

In [ ]:
final_data = clean_and_merge_climate_data(energy_file, pm25_file)

## Step 4 - Take a quick look at the cleaned data

It is good practice to look at the data before using it. Each row is one country in one year, with:

- `% GHG emitting source` - share of electricity from coal, gas and oil
- `% GHG non emitting source` - share of electricity from clean sources
- `PM2.5 Exposure` - average air pollution people in that country were exposed to (µg/m³)

In [ ]:
final_data.head(10)

In [ ]:
# Summary statistics: count, average, smallest and largest values for each column
final_data.describe()

# Part 2 - Teaching a computer to predict pollution

## Step 5 - Load the cleaned dataset and choose inputs and output

We reload the cleaned file, as a separate project might have done. Then we tell the model what it is allowed to look at and what it must predict:

- **Features (`X`)** - the two electricity percentages. These are the *clues*.
- **Target (`y`)** - PM2.5 exposure. This is the *answer* we want the model to guess.

In [ ]:
df = pd.read_csv('cleaned_climate_merged.csv')

X = df[['% GHG emitting source', '% GHG non emitting source']]   # the clues
y = df['PM2.5 Exposure']                                          # the answer to predict

print(f"Rows of data available: {len(df)}")

## Step 6 - Split the data into a study set and an exam set

Imagine a student who memorises last year's exam answers. They might score 100% on that exam but still not understand the subject. To avoid fooling ourselves, we do the same as a good teacher would:

- **Training set (80%)** - the "study material" the model learns from.
- **Testing set (20%)** - a hidden "exam" the model has never seen. How well it does here tells us how well it would do on new, real-world cases.

The split is random, but `random_state=42` fixes the shuffle so anyone running this notebook gets exactly the same split and the same results.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Number of training samples: {len(X_train)}")
print(f"Number of testing samples:  {len(X_test)}")

## Step 7 - Train the model (a "Random Forest")

We use a model called a **Random Forest**. Think of it as asking **100 different experts** for their opinion and then averaging their answers.

- Each expert is a *decision tree*, which is like a flowchart of yes/no questions ("Is more than 50% of the electricity from coal? If yes, ...").
- Each tree sees a slightly different slice of the data, so they make different mistakes.
- Averaging all 100 predictions smooths out those mistakes and gives a more reliable answer.

`n_estimators=100` means "use 100 trees". **Training** (`fit`) is the step where the trees study the training set and work out how the electricity mix relates to pollution.

In [ ]:
print("Training the model...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Done!")

## Step 8 - Make predictions on the hidden "exam"

Now we hand the model only the clues (the electricity percentages) from the testing set and ask it to guess the pollution. We keep the real answers (`y_test`) to ourselves so we can grade it next.

In [ ]:
y_pred = model.predict(X_test)

## Step 9 - Grade the model

We compare the model's guesses with the real answers using three standard scores:

| Score | What it tells you | Good value |
|---|---|---|
| **RMSE** (Root Mean Squared Error) | The typical size of the model's mistakes, in µg/m³. It punishes big mistakes more heavily. | Lower is better |
| **MAE** (Mean Absolute Error) | The plain average size of the mistakes, in µg/m³. For example, an MAE of 5 means the guesses are off by about 5 on average. | Lower is better |
| **R² (R-squared)** | The share of the ups and downs in pollution that the model can explain. 1.0 is perfect; 0 means no better than always guessing the average; negative means worse than that. | Closer to 1 is better |

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print("Model Evaluation Results:")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f} µg/m³")
print(f"MAE (Mean Absolute Error): {mae:.2f} µg/m³")
print(f"R-squared (R²): {r2:.4f}")
print("-" * 30)

## Step 10 - See the results as a picture

Each blue dot is one country-year from the exam set:

- The **horizontal position** is the real pollution level.
- The **vertical position** is what the model predicted.

The **red dashed line** is what a *perfect* model would produce (prediction = reality). The closer the dots hug that line, the better the model. Dots far above the line are over-predictions; dots far below are under-predictions.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='blue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)  # the "perfect prediction" line
plt.xlabel('Actual PM2.5 Exposure')
plt.ylabel('Predicted PM2.5 Exposure')
plt.title('Actual vs Predicted PM2.5 (Random Forest)')
plt.grid(True)
plt.show()

## Step 11 - Which clue mattered more?

A Random Forest can tell us how much it leaned on each input when making its guesses. These **feature importances** add up to 100%. A larger number means that input was more useful for predicting pollution.

> **Be careful with the conclusion.** A model finding a pattern does not prove that one thing *causes* the other. Richer countries, for example, tend to differ in many ways at once (industry, transport, climate, geography), and we have only given the model two inputs. Treat the results as a clue, not a verdict.

In [ ]:
importances = model.feature_importances_
print("Feature Importances:")
print(f"- GHG emitting source: {importances[0]*100:.2f}%")
print(f"- GHG non emitting source: {importances[1]*100:.2f}%")

## Summary

- We combined **electricity-mix data** with **air-pollution data** and cleaned it, keeping only countries with at least 15 usable years of records.
- We trained a **Random Forest** on 80% of the data and tested it on the other 20%.
- The scores in Step 9 and the chart in Step 10 show how closely the electricity mix alone can predict PM2.5 pollution.
- The importances in Step 11 show which of the two inputs the model relied on more.

**Ideas for next steps:** add more inputs (population, income, industry, weather), try other models, or check one country at a time.